In [1]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import os

DATA = '/kaggle/input/competitions/march-machine-learning-mania-2026'

for f in sorted(os.listdir(DATA)):
    size = os.path.getsize(f'{DATA}/{f}') / 1024
    print(f"{f:<50} {size:>10.1f} KB")

Cities.csv                                                9.5 KB
Conferences.csv                                           1.6 KB
MConferenceTourneyGames.csv                             174.8 KB
MGameCities.csv                                        2854.3 KB
MMasseyOrdinals.csv                                  123820.7 KB
MNCAATourneyCompactResults.csv                           75.9 KB
MNCAATourneyDetailedResults.csv                         141.5 KB
MNCAATourneySeedRoundSlots.csv                           15.5 KB
MNCAATourneySeeds.csv                                    38.6 KB
MNCAATourneySlots.csv                                    50.5 KB
MRegularSeasonCompactResults.csv                       5632.8 KB
MRegularSeasonDetailedResults.csv                     11927.3 KB
MSeasons.csv                                              1.8 KB
MSecondaryTourneyCompactResults.csv                      62.1 KB
MSecondaryTourneyTeams.csv                               27.8 KB
MTeamCoaches.csv         

In [2]:
# Men
MTeams    = pd.read_csv(f'{DATA}/MTeams.csv')
MRegular  = pd.read_csv(f'{DATA}/MRegularSeasonDetailedResults.csv')
MTourney  = pd.read_csv(f'{DATA}/MNCAATourneyCompactResults.csv')
MSeeds    = pd.read_csv(f'{DATA}/MNCAATourneySeeds.csv')
MMassey   = pd.read_csv(f'{DATA}/MMasseyOrdinals.csv')

# Women
WTeams    = pd.read_csv(f'{DATA}/WTeams.csv')
WRegular  = pd.read_csv(f'{DATA}/WRegularSeasonDetailedResults.csv')
WTourney  = pd.read_csv(f'{DATA}/WNCAATourneyCompactResults.csv')
WSeeds    = pd.read_csv(f'{DATA}/WNCAATourneySeeds.csv')

# Submission target
Sub = pd.read_csv(f'{DATA}/SampleSubmissionStage2.csv')

print("MRegular :", MRegular.shape)
print("MTourney :", MTourney.shape)
print("MSeeds   :", MSeeds.shape)
print("MMassey  :", MMassey.shape)
print("WRegular :", WRegular.shape)
print("WTourney :", WTourney.shape)
print("WSeeds   :", WSeeds.shape)
print("Sub      :", Sub.shape)
print("\nSub sample:")
print(Sub.head(3))

MRegular : (122775, 34)
MTourney : (2585, 8)
MSeeds   : (2626, 3)
MMassey  : (5761702, 5)
WRegular : (85505, 34)
WTourney : (1717, 8)
WSeeds   : (1744, 3)
Sub      : (132133, 2)

Sub sample:
               ID  Pred
0  2026_1101_1102   0.5
1  2026_1101_1103   0.5
2  2026_1101_1104   0.5


In [3]:
def compute_season_stats(df):
    records = []
    for season, grp in df.groupby('Season'):
        teams = set(grp['WTeamID']) | set(grp['LTeamID'])
        for team in teams:
            w = grp[grp['WTeamID'] == team]
            l = grp[grp['LTeamID'] == team]
            g = len(w) + len(l)
            if g == 0: continue

            pts_for  = w['WScore'].sum() + l['LScore'].sum()
            pts_agn  = w['LScore'].sum()  + l['WScore'].sum()

            fgm  = w['WFGM'].sum()  + l['LFGM'].sum()
            fga  = w['WFGA'].sum()  + l['LFGA'].sum()
            fgm3 = w['WFGM3'].sum() + l['LFGM3'].sum()
            fga3 = w['WFGA3'].sum() + l['LFGA3'].sum()
            ftm  = w['WFTM'].sum()  + l['LFTM'].sum()
            fta  = w['WFTA'].sum()  + l['LFTA'].sum()
            reb  = w['WOR'].sum() + w['WDR'].sum() + l['LOR'].sum() + l['LDR'].sum()
            ast  = w['WAst'].sum() + l['LAst'].sum()
            to   = w['WTO'].sum()  + l['LTO'].sum()
            stl  = w['WStl'].sum() + l['LStl'].sum()
            blk  = w['WBlk'].sum() + l['LBlk'].sum()

            records.append({
                'Season'     : season,
                'TeamID'     : team,
                'WinRate'    : len(w) / g,
                'AvgPtsFor'  : pts_for / g,
                'AvgPtsAgn'  : pts_agn / g,
                'AvgPtsDiff' : (pts_for - pts_agn) / g,
                'FGPct'      : fgm  / fga  if fga  > 0 else 0,
                'FG3Pct'     : fgm3 / fga3 if fga3 > 0 else 0,
                'FTPct'      : ftm  / fta  if fta  > 0 else 0,
                'AvgReb'     : reb / g,
                'AvgAst'     : ast / g,
                'AvgTO'      : to  / g,
                'AvgStl'     : stl / g,
                'AvgBlk'     : blk / g,
                'Games'      : g,
            })
    return pd.DataFrame(records)

print("Computing Men's stats...")
M_stats = compute_season_stats(MRegular)

print("Computing Women's stats...")
W_stats = compute_season_stats(WRegular)

print("M_stats:", M_stats.shape)
print("W_stats:", W_stats.shape)
print("\nTop 5 Men 2025 by WinRate:")
print(M_stats[M_stats['Season']==2025].sort_values('WinRate', ascending=False).head(5)[['TeamID','WinRate','AvgPtsDiff','FGPct']].to_string())

Computing Men's stats...
Computing Women's stats...
M_stats: (8346, 15)
W_stats: (5965, 15)

Top 5 Men 2025 by WinRate:
      TeamID   WinRate  AvgPtsDiff     FGPct
7691    1181  0.911765   20.794118  0.488060
7689    1179  0.903226    8.967742  0.473752
7706    1196  0.882353   16.176471  0.472669
7888    1385  0.882353   12.823529  0.450831
7730    1222  0.882353   15.735294  0.458229


In [4]:
def compute_elo(results_df, k=20, base=1500):
    season_elo = {}

    for season, grp in results_df.groupby('Season'):
        # Carry 50% of prior season rating deviation into new season
        if season_elo:
            prev = season_elo[max(s for s in season_elo if s < season)]
            elo  = {t: base + 0.5*(r - base) for t, r in prev.items()}
        else:
            elo = {}

        for _, row in grp.sort_values('DayNum').iterrows():
            w      = row['WTeamID']
            l      = row['LTeamID']
            margin = row['WScore'] - row['LScore']

            ew = elo.get(w, base)
            el = elo.get(l, base)

            exp_w = 1 / (1 + 10**((el - ew) / 400))

            # Margin of victory multiplier — log scaled, capped
            mov   = np.clip(np.log1p(abs(margin)) / np.log1p(20), 0.5, 2.5)
            k_adj = k * mov

            elo[w] = ew + k_adj * (1 - exp_w)
            elo[l] = el + k_adj * (0 - (1 - exp_w))

        season_elo[season] = elo

    return season_elo

print("Computing Men's Elo...")
M_elo = compute_elo(MRegular)

print("Computing Women's Elo...")
W_elo = compute_elo(WRegular)

# Sanity check — top 8 men's teams in 2025
print("\nTop 8 Men 2025 by Elo:")
top = sorted(M_elo[2025].items(), key=lambda x: -x[1])[:8]
for tid, rating in top:
    name = MTeams[MTeams['TeamID']==tid]['TeamName'].values[0]
    print(f"  {name:<25} {rating:.1f}")

Computing Men's Elo...
Computing Women's Elo...

Top 8 Men 2025 by Elo:
  Houston                   1779.4
  Duke                      1765.9
  Florida                   1747.0
  Auburn                    1738.9
  St John's                 1721.0
  Gonzaga                   1710.2
  Tennessee                 1709.3
  St Mary's CA              1708.1


In [5]:
import re

def clean_seeds(seeds_df):
    df = seeds_df.copy()
    df['SeedNum'] = df['Seed'].apply(lambda s: int(re.findall(r'\d+', str(s))[0]))
    return df[['Season','TeamID','SeedNum']]

def compute_massey(massey_df, cutoff=133):
    # Only use rankings right before tournament
    late = massey_df[massey_df['RankingDayNum'] <= cutoff]
    agg  = (late.groupby(['Season','TeamID'])['OrdinalRank']
                .agg(MeanRank='mean', MinRank='min', MaxRank='max')
                .reset_index())
    return agg

MSeeds_clean = clean_seeds(MSeeds)
WSeeds_clean = clean_seeds(WSeeds)
M_massey     = compute_massey(MMassey)

print("MSeeds_clean:", MSeeds_clean.shape)
print("WSeeds_clean:", WSeeds_clean.shape)
print("M_massey    :", M_massey.shape)

print("\nTop 10 Men 2025 by Massey MeanRank:")
print(M_massey[M_massey['Season']==2025]
      .sort_values('MeanRank').head(10)
      [['TeamID','MeanRank','MinRank']].to_string())

MSeeds_clean: (2626, 3)
WSeeds_clean: (1744, 3)
M_massey    : (8356, 5)

Top 10 Men 2025 by Massey MeanRank:
      TeamID   MeanRank  MinRank
7644    1120   1.722096        1
7701    1181   4.817768        1
7910    1397   4.831435        1
7630    1104   6.415718        1
7716    1196   6.797267        1
7740    1222   9.751432        1
7753    1235  10.627563        1
7764    1246  14.627721        1
7860    1345  15.325714        1
7760    1242  15.945665        1


In [6]:
def compute_recent_form(df, days=30):
    records = []
    for season, grp in df.groupby('Season'):
        cutoff = grp['DayNum'].max() - days
        recent = grp[grp['DayNum'] >= cutoff]
        teams  = set(grp['WTeamID']) | set(grp['LTeamID'])

        for team in teams:
            w = recent[recent['WTeamID'] == team]
            l = recent[recent['LTeamID'] == team]
            g = len(w) + len(l)

            if g == 0:
                rwr, rpd = 0.5, 0.0
            else:
                pts_f = w['WScore'].sum() + l['LScore'].sum()
                pts_a = w['LScore'].sum() + l['WScore'].sum()
                rwr   = len(w) / g
                rpd   = (pts_f - pts_a) / g

            records.append({
                'Season'        : season,
                'TeamID'        : team,
                'RecentWinRate' : rwr,
                'RecentPtsDiff' : rpd,
                'RecentGames'   : g
            })
    return pd.DataFrame(records)

print("Computing Men's recent form...")
M_recent = compute_recent_form(MRegular)

print("Computing Women's recent form...")
W_recent = compute_recent_form(WRegular)

print("M_recent:", M_recent.shape)
print("W_recent:", W_recent.shape)

print("\nTop 5 Men 2025 by Recent Win Rate:")
print(M_recent[M_recent['Season']==2025]
      .sort_values('RecentWinRate', ascending=False)
      .head(5)[['TeamID','RecentWinRate','RecentPtsDiff','RecentGames']].to_string())

Computing Men's recent form...
Computing Women's recent form...
M_recent: (8346, 5)
W_recent: (5965, 5)

Top 5 Men 2025 by Recent Win Rate:
      TeamID  RecentWinRate  RecentPtsDiff  RecentGames
7671    1161            1.0      16.900000           10
7691    1181            1.0      23.700000           10
7971    1471            1.0      19.888889            9
7778    1270            1.0      16.500000            8
7857    1352            1.0      11.142857            7


In [7]:
STAT_COLS   = ['WinRate','AvgPtsFor','AvgPtsAgn','AvgPtsDiff',
               'FGPct','FG3Pct','FTPct','AvgReb','AvgAst','AvgTO','AvgStl','AvgBlk','Games']
RECENT_COLS = ['RecentWinRate','RecentPtsDiff']

FEATURE_COLS = (
    ['EloDiff','SeedDiff','MeanRankDiff'] +
    [f'T1_{c}' for c in STAT_COLS] +
    [f'T2_{c}' for c in STAT_COLS] +
    ['WinRateDiff','PtsDiffDiff'] +
    [f'T1_{c}' for c in RECENT_COLS] +
    [f'T2_{c}' for c in RECENT_COLS] +
    ['RecentWinRateDiff','RecentPtsDiffDiff'] +
    ['T1_SeedNum','T2_SeedNum','T1_MeanRank','T2_MeanRank','T1_MinRank','T2_MinRank']
)

def build_dataset(tourney_df, stats_df, seeds_df, massey_df, recent_df, elo_dict, with_massey=True):
    rows = []
    for _, row in tourney_df.iterrows():
        w, l   = row['WTeamID'], row['LTeamID']
        t1, t2 = (w, l) if w < l else (l, w)
        rows.append({'Season': row['Season'], 'Team1': t1, 'Team2': t2,
                     'Label': 1 if w == t1 else 0})
    df = pd.DataFrame(rows)

    # Elo
    elo_rows = [{'Season': s, 'TeamID': t, 'Elo': r}
                for s, ed in elo_dict.items() for t, r in ed.items()]
    elo_df = pd.DataFrame(elo_rows)
    for prefix, col in [('T1','Team1'),('T2','Team2')]:
        df = df.merge(elo_df.rename(columns={'TeamID': col, 'Elo': f'{prefix}_Elo'}),
                      on=['Season', col], how='left')
    df['T1_Elo'] = df['T1_Elo'].fillna(1500)
    df['T2_Elo'] = df['T2_Elo'].fillna(1500)
    df['EloDiff'] = df['T1_Elo'] - df['T2_Elo']

    # Season stats
    for prefix, col in [('T1','Team1'),('T2','Team2')]:
        tmp = stats_df.rename(columns={'TeamID': col})[['Season', col] + STAT_COLS]
        tmp = tmp.rename(columns={c: f'{prefix}_{c}' for c in STAT_COLS})
        df  = df.merge(tmp, on=['Season', col], how='left')

    # Seeds
    for prefix, col in [('T1','Team1'),('T2','Team2')]:
        tmp = seeds_df.rename(columns={'TeamID': col})[['Season', col, 'SeedNum']]
        tmp = tmp.rename(columns={'SeedNum': f'{prefix}_SeedNum'})
        df  = df.merge(tmp, on=['Season', col], how='left')
    df['SeedDiff'] = df['T1_SeedNum'] - df['T2_SeedNum']

    # Massey
    if with_massey and massey_df is not None:
        for prefix, col in [('T1','Team1'),('T2','Team2')]:
            tmp = massey_df.rename(columns={'TeamID': col})[['Season', col, 'MeanRank','MinRank']]
            tmp = tmp.rename(columns={'MeanRank': f'{prefix}_MeanRank',
                                      'MinRank' : f'{prefix}_MinRank'})
            df  = df.merge(tmp, on=['Season', col], how='left')
        df['MeanRankDiff'] = df['T1_MeanRank'] - df['T2_MeanRank']
    else:
        df['T1_MeanRank']=175; df['T2_MeanRank']=175
        df['T1_MinRank'] =175; df['T2_MinRank'] =175
        df['MeanRankDiff']=0

    # Recent form
    for prefix, col in [('T1','Team1'),('T2','Team2')]:
        tmp = recent_df.rename(columns={'TeamID': col})[['Season', col] + RECENT_COLS]
        tmp = tmp.rename(columns={c: f'{prefix}_{c}' for c in RECENT_COLS})
        df  = df.merge(tmp, on=['Season', col], how='left')

    # Diff features
    df['WinRateDiff']       = df['T1_WinRate']      - df['T2_WinRate']
    df['PtsDiffDiff']       = df['T1_AvgPtsDiff']   - df['T2_AvgPtsDiff']
    df['RecentWinRateDiff'] = df['T1_RecentWinRate'] - df['T2_RecentWinRate']
    df['RecentPtsDiffDiff'] = df['T1_RecentPtsDiff'] - df['T2_RecentPtsDiff']

    # Fill missing
    for c in df.columns:
        if 'Rank' in c: df[c] = df[c].fillna(175)
        elif 'Seed' in c: df[c] = df[c].fillna(16)
    df[FEATURE_COLS] = df[FEATURE_COLS].fillna(df[FEATURE_COLS].median())

    return df

print("Building Men's training data...")
M_train = build_dataset(MTourney, M_stats, MSeeds_clean, M_massey, M_recent, M_elo, with_massey=True)

print("Building Women's training data...")
W_train = build_dataset(WTourney, W_stats, WSeeds_clean, None, W_recent, W_elo, with_massey=False)

print(f"\nM_train : {M_train.shape}  | nulls: {M_train[FEATURE_COLS].isnull().sum().sum()}")
print(f"W_train : {W_train.shape}  | nulls: {W_train[FEATURE_COLS].isnull().sum().sum()}")
print(f"Features: {len(FEATURE_COLS)}")

Building Men's training data...
Building Women's training data...

M_train : (2585, 49)  | nulls: 0
W_train : (1717, 49)  | nulls: 0
Features: 43


In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import brier_score_loss, log_loss
import xgboost as xgb
import lightgbm as lgb

def train_ensemble(X, y, n_splits=5, seed=42):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)

    oof_xgb = np.zeros(len(y))
    oof_lgb = np.zeros(len(y))
    oof_lr  = np.zeros(len(y))

    xgb_models, lgb_models, lr_models = [], [], []

    for fold, (tr, val) in enumerate(skf.split(X, y)):
        Xtr, Xval = X.iloc[tr], X.iloc[val]
        ytr, yval = y[tr], y[val]

        # XGBoost
        m_xgb = xgb.XGBClassifier(
            n_estimators=600, learning_rate=0.02, max_depth=4,
            subsample=0.8, colsample_bytree=0.8,
            eval_metric='logloss', verbosity=0, random_state=seed)
        m_xgb.fit(Xtr, ytr, eval_set=[(Xval, yval)], verbose=False)
        oof_xgb[val] = m_xgb.predict_proba(Xval)[:,1]
        xgb_models.append(m_xgb)

        # LightGBM
        m_lgb = lgb.LGBMClassifier(
            n_estimators=600, learning_rate=0.02, max_depth=4,
            subsample=0.8, colsample_bytree=0.8,
            random_state=seed, verbose=-1)
        m_lgb.fit(Xtr, ytr, eval_set=[(Xval, yval)],
                  callbacks=[lgb.early_stopping(50, verbose=False),
                             lgb.log_evaluation(-1)])
        oof_lgb[val] = m_lgb.predict_proba(Xval)[:,1]
        lgb_models.append(m_lgb)

        # Logistic Regression
        sc   = StandardScaler()
        m_lr = LogisticRegression(C=0.1, max_iter=1000, random_state=seed)
        m_lr.fit(sc.fit_transform(Xtr), ytr)
        oof_lr[val] = m_lr.predict_proba(sc.transform(Xval))[:,1]
        lr_models.append((sc, m_lr))

        ens = (oof_xgb[val] + oof_lgb[val] + oof_lr[val]) / 3
        print(f"  Fold {fold+1} | Brier: {brier_score_loss(yval, ens):.5f} | LogLoss: {log_loss(yval, ens):.5f}")

    oof_ens = (oof_xgb + oof_lgb + oof_lr) / 3
    print(f"\n  OOF Brier  : {brier_score_loss(y, oof_ens):.5f}")
    print(f"  OOF LogLoss: {log_loss(y, oof_ens):.5f}")

    return xgb_models, lgb_models, lr_models

X_men   = M_train[FEATURE_COLS]
y_men   = M_train['Label'].values
X_women = W_train[FEATURE_COLS]
y_women = W_train['Label'].values

print("=" * 45)
print("Training Men's Models")
print("=" * 45)
M_xgb, M_lgb, M_lr = train_ensemble(X_men, y_men)

print()
print("=" * 45)
print("Training Women's Models")
print("=" * 45)
W_xgb, W_lgb, W_lr = train_ensemble(X_women, y_women)

Training Men's Models
  Fold 1 | Brier: 0.19294 | LogLoss: 0.56123
  Fold 2 | Brier: 0.18229 | LogLoss: 0.54004
  Fold 3 | Brier: 0.18534 | LogLoss: 0.54322
  Fold 4 | Brier: 0.18584 | LogLoss: 0.54960
  Fold 5 | Brier: 0.18835 | LogLoss: 0.55394

  OOF Brier  : 0.18695
  OOF LogLoss: 0.54961

Training Women's Models
  Fold 1 | Brier: 0.15069 | LogLoss: 0.45685
  Fold 2 | Brier: 0.15015 | LogLoss: 0.44527
  Fold 3 | Brier: 0.14502 | LogLoss: 0.43241
  Fold 4 | Brier: 0.13699 | LogLoss: 0.41784
  Fold 5 | Brier: 0.15262 | LogLoss: 0.45588

  OOF Brier  : 0.14710
  OOF LogLoss: 0.44166


In [9]:
def predict(X, xgb_models, lgb_models, lr_models):
    n   = len(xgb_models)
    p   = np.zeros(len(X))
    for m in xgb_models:
        p += m.predict_proba(X)[:,1] / n
    for m in lgb_models:
        p += m.predict_proba(X)[:,1] / n
    for sc, m in lr_models:
        p += m.predict_proba(sc.transform(X))[:,1] / n
    return p / 3


def build_submission(sub_df, stats_df, seeds_df, massey_df, recent_df,
                     elo_dict, xgb_m, lgb_m, lr_m, gender_prefix, with_massey=True):

    df = sub_df.copy()
    df[['Season','Team1','Team2']] = df['ID'].str.split('_', expand=True).astype(int)

    if gender_prefix == 'M':
        df = df[(df['Team1'] < 3000) & (df['Team2'] < 3000)]
    else:
        df = df[(df['Team1'] >= 3000) & (df['Team2'] >= 3000)]

    # Elo
    elo_rows = [{'Season': s, 'TeamID': t, 'Elo': r}
                for s, ed in elo_dict.items() for t, r in ed.items()]
    elo_df = pd.DataFrame(elo_rows)
    for prefix, col in [('T1','Team1'),('T2','Team2')]:
        df = df.merge(elo_df.rename(columns={'TeamID': col, 'Elo': f'{prefix}_Elo'}),
                      on=['Season', col], how='left')
    df['T1_Elo'] = df['T1_Elo'].fillna(1500)
    df['T2_Elo'] = df['T2_Elo'].fillna(1500)
    df['EloDiff'] = df['T1_Elo'] - df['T2_Elo']

    # Season stats
    for prefix, col in [('T1','Team1'),('T2','Team2')]:
        tmp = stats_df.rename(columns={'TeamID': col})[['Season', col] + STAT_COLS]
        tmp = tmp.rename(columns={c: f'{prefix}_{c}' for c in STAT_COLS})
        df  = df.merge(tmp, on=['Season', col], how='left')

    # Seeds
    for prefix, col in [('T1','Team1'),('T2','Team2')]:
        tmp = seeds_df.rename(columns={'TeamID': col})[['Season', col, 'SeedNum']]
        tmp = tmp.rename(columns={'SeedNum': f'{prefix}_SeedNum'})
        df  = df.merge(tmp, on=['Season', col], how='left')
    df['SeedDiff'] = df['T1_SeedNum'] - df['T2_SeedNum']

    # Massey
    if with_massey and massey_df is not None:
        for prefix, col in [('T1','Team1'),('T2','Team2')]:
            tmp = massey_df.rename(columns={'TeamID': col})[['Season', col, 'MeanRank','MinRank']]
            tmp = tmp.rename(columns={'MeanRank': f'{prefix}_MeanRank',
                                      'MinRank' : f'{prefix}_MinRank'})
            df  = df.merge(tmp, on=['Season', col], how='left')
        df['MeanRankDiff'] = df['T1_MeanRank'] - df['T2_MeanRank']
    else:
        df['T1_MeanRank']=175; df['T2_MeanRank']=175
        df['T1_MinRank'] =175; df['T2_MinRank'] =175
        df['MeanRankDiff']=0

    # Recent form
    for prefix, col in [('T1','Team1'),('T2','Team2')]:
        tmp = recent_df.rename(columns={'TeamID': col})[['Season', col] + RECENT_COLS]
        tmp = tmp.rename(columns={c: f'{prefix}_{c}' for c in RECENT_COLS})
        df  = df.merge(tmp, on=['Season', col], how='left')

    # Diffs
    df['WinRateDiff']       = df['T1_WinRate']      - df['T2_WinRate']
    df['PtsDiffDiff']       = df['T1_AvgPtsDiff']   - df['T2_AvgPtsDiff']
    df['RecentWinRateDiff'] = df['T1_RecentWinRate'] - df['T2_RecentWinRate']
    df['RecentPtsDiffDiff'] = df['T1_RecentPtsDiff'] - df['T2_RecentPtsDiff']

    # Fill
    for c in df.columns:
        if 'Rank' in c: df[c] = df[c].fillna(175)
        elif 'Seed' in c: df[c] = df[c].fillna(16)
    df[FEATURE_COLS] = df[FEATURE_COLS].fillna(0)

    df['Pred'] = np.clip(predict(df[FEATURE_COLS], xgb_m, lgb_m, lr_m), 0.025, 0.975)
    return df[['ID','Pred']]


print("Building Men's submission...")
M_sub = build_submission(Sub, M_stats, MSeeds_clean, M_massey, M_recent,
                          M_elo, M_xgb, M_lgb, M_lr, gender_prefix='M', with_massey=True)

print("Building Women's submission...")
W_sub = build_submission(Sub, W_stats, WSeeds_clean, None, W_recent,
                          W_elo, W_xgb, W_lgb, W_lr, gender_prefix='W', with_massey=False)

# Combine
final = pd.concat([M_sub, W_sub], ignore_index=True)
final = Sub[['ID']].merge(final, on='ID', how='left')
final['Pred'] = final['Pred'].fillna(0.5).clip(0.025, 0.975)

final.to_csv('/kaggle/working/submission.csv', index=False)

print(f"\nShape : {final.shape}")
print(f"Nulls : {final['Pred'].isnull().sum()}")
print(f"Min   : {final['Pred'].min():.4f}")
print(f"Max   : {final['Pred'].max():.4f}")
print(f"Mean  : {final['Pred'].mean():.4f}")
print("\n✅ Saved to /kaggle/working/submission.csv")

Building Men's submission...
Building Women's submission...

Shape : (132133, 2)
Nulls : 0
Min   : 0.0250
Max   : 0.6982
Mean  : 0.2457

✅ Saved to /kaggle/working/submission.csv
